In [1]:
%load_ext jupyter_turtle
import jupyter_turtle as tu

# Interactivity

Functions are a powerful tool in programming. Not only do they give you a way to reuse and parameterize code but they open the possibility of *event driven* programming. Event driven programming is when a program responds to real world events as they happen. Examples of events are a mouse click or key press. Event driven programs are *interactive* because they react to input immediately. In this lesson you'll learn how to use widgets to provide input to a function.

To use interactivity you have to import `ipywidgets`:

In [2]:
import ipywidgets as widgets

## Interacting with Functions

The `ipywidgets.interact` *decorator* makes it easy to connect a function to interactive controls in a notebook. Instead of having to call a function directly, you can let the notebook create widgets for you and call the function whenever the value changes. This is a simple way to turn a regular function into a small interactive program.

Here's a function that prints its argument. What's different than what you've seen before is that there's a decorator above the function that makes it interactive. 

In [4]:
@widgets.interact(a=100) # This is a decorator
def hello_interact(a):
    """Return a string that describes a."""
    return f"""a: type: {type(a)}: value: {a}"""

interactive(children=(IntSlider(value=100, description='a', max=300, min=-100), Output()), _dom_classes=('widg…

Decorators are a way to change or enhance a function without rewriting the function itself. In Python, a decorator is written with the `@` symbol and placed above a function definition. It lets you add behavior such as interactive controls in a clean and reusable way. The `ipywidgets.interact` decorator takes one argument for each argument of the function that it decorates. The argument provides a default value for the decorated function as well as a default *type*. The type is used to determine what widget to use. 

Try changing the default value of the `a` argument in the next cell to the following things:

1. A `float` (e.g. `1.1`)
2. A `str` (e.g. `"Hello"`)
3. A `bool` (e.g. `True`)
4. A list (e.g. `["Eat", "Sleep", "Program", "Repeat"]`)

Notice how the type of the default argument controls the widget:

In [8]:
@widgets.interact(a=10) 
def hello_interact(a):
    """Return a string that describes a."""
    return f"""a: type: {type(a)}: value: {a}"""

interactive(children=(IntSlider(value=10, description='a', max=30, min=-10), Output()), _dom_classes=('widget-…

## Designing With Widgets

Widgets make it easy to call a function without having to write the function call. They make it so that the people who use your Jupyter Notebook don't have to be programmers. Widgets also let you set limits on the values that can be chosen and pick the most useful interface for an input.

Let's see examples of how widgets are customized, starting with a `shape` function that draws an arbitrary shape using the Turtle: 

```python
@widgets.interact(size=50, sides=3, turns=1)
def shape(size, sides, turns):
    tu.clear() # Clear the previous drawing
    for _ in range(sides):
        tu.turn(360*turns/sides)
        tu.move(size)
```

Copy the example into the next cell:

**It has problems!** The `tu.clear()` function clears the drawing but doesn't reset the Turtle's position. If you give shape a bad combination of inputs it causes the turtle to run off of the screen and never come back! 

We need to set limits so that it's impossible to draw a weird shape. Notice in the next example that instead of a single default we provide an acceptable range for each input.

```python
@widgets.interact(size=(0,150), sides=(3,10), turns=(1,2))
def shape(size, sides, turns):
    tu.clear()
    for _ in range(sides):
        tu.turn(360*turns/sides)
        tu.move(size)
```

Paste the updated code below:

### Draw a Star 

Yay! Let's use the `shape` function as the basis for doing other things with the turtle. If we call shape with `sides=5` and `turns=2` it draws a star. Let's draw a star and make it possible to rotate in any direction from -180 to 180 degrees. Also, let's let the user select if they want to hide the Turtle for the drawing.

Here's a new function `heading_star` that adds a `heading` parameter and a boolean to control the turtle visibility:

```python
@widgets.interact(size=(0,150), heading=(-180,180), hide=False)
def heading_star(size, heading, hide):
    if hide:
        tu.hide_turtle()
    else:
        tu.show_turtle()
    tu.set_heading(heading)
    shape(size, 5, 2)
```

### Add Writing

Got text? Let's give ourselves a star using the `heading_star` function. We'll add some text to the drawing and pick a color from a list of choices. 

```python
@widgets.interact(heading=(-180,180), text="Hello World", color=["black", "red", "green", "blue"])
def word_star(heading, text, color):
    tu.set_color(color)
    heading_star(100, heading, True)
    tu.write(text)
```

### Choose Your Own Color

More colors! There are millions of possible colors that the computer is able to reproduce. What if we let our user pick any color they want? That's the job of a the `ColorPicker` widget. Let's update the last example to use that widget instead of a fixed list:

```python
@widgets.interact(heading=(-180,180), text="Hello World", color=widgets.ColorPicker())
def word_star(heading, text, color):
    tu.set_color(color)
    heading_star(100, heading, True)
    tu.write(text)
```

## Reusing a Function 

In all of our previous examples we rewrote the function every time we wanted to make it interactive. What if we had a function from someone or somewhere else that we wanted to make interactive? Instead of using `interact` as a decorator, we can use it as a plain old function like this:

```python 
widgets.interact(shape, size=(0,150), sides=(3,12), turns=(1,2))
```

**Notice how, when used as a function, `interact` requires the name of the target function as its first argument.**

### Customize a Widget's Appearance

The default widget shows the name of the variable as the help text on the left. It's possible to customize widgets in many different ways. Here's an update of the last example where we create the widgets in advance. This has some important implications, as we'll see a bit later. 

```python
heading_widget = widgets.IntSlider(description="Heading", min=-180, max=180)
text_widget = widgets.Text(description="Name", placeholder="Your Name Here")
color_widget = widgets.ColorPicker(description="Color") 

widgets.interact(word_star, heading=heading_widget, text=text_widget, color=color_widget)
```

Try the example in the next cell:

You can find a completelist of available widgets on the [Jupyter Widgets website](https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20List.html).

## Global Widgets 

Widgets created outside of a funtion are global. That means they can be accessed from inside of the function as well as other parts of the program. This is necessary if you want to be able to change a widgets appearance in response to some action. Let's see an example.

Suppose you had a UI for ordering at a coffee shop. You can order an item and select add-ons for the item. The valid add-ons depend on the item. For example, coffee might have half-and-half, soy and almond milk and a bagel might have butter, cream cheese and lox. You don't want the user to be able to select lox for their coffee! 

Here's an example UI:

In [82]:
# Global widget variables 
item_widget = widgets.Dropdown(description='Item:', options=["Coffee", "Bagel"])
addon_widget = widgets.RadioButtons(description='Add:')

# Interactive function
def update_item(item):
    """Update the options when the item value changes"""
    global item_widget, addon_widget
    if item == "Coffee":
        addon_widget.description = "Creamer"
        addon_widget.options = ["Half and Half", "Soy Milk", "Oat Milk"]
    elif item == "Bagel":
        addon_widget.description = "Topping"
        addon_widget.options = ["Butter", "Cream Cheese", "Lox"]

# Display the function and the addon widgets:
display(
    widgets.interactive(update_item, item=item_widget), 
    addon_widget
)

interactive(children=(Dropdown(description='Item:', options=('Coffee', 'Bagel'), value='Coffee'), Output()), _…

RadioButtons(description='Creamer', options=('Half and Half', 'Soy Milk', 'Oat Milk'), value=None)

## What is a Decorator? 

A decorator takes one function as input, creates a new function that adds extra behavior, and then returns that new function. This makes decorators a powerful tool for modifying how functions work while keeping the original code readable. You don't have to write your own decorator, but here's an example of a function that's also a decorator:

In [83]:
def alert(func):
    """Wrap the function func() with print statements"""
    
    # Create a function that wrapps around func()
    def wrapper():
        print("Before")
        func()
        print("After")

    # A decorator takes a function as an argument and returns a function!
    return wrapper

Run the code in the previous cell and you'll be able to use it as a decorator. 

```python 
@alert
def my_function():
    print("This is my function")
```

Now try calling `my_function`:

```python
my_function()
```